# Phase 3A — SwinUNETR Fine-tuning & Spatial Embedding (BraTS 2024)
## 3 output channels: WT / TC / ET (Correct BraTS 2024 labels: ET=3, RC=4 excluded)

**Pretrained**: SwinUNETR BraTS 2021 Fold 1 (Mean Dice = 0.9059)

Fine-tuning on BraTS 2024 Post-Treatment with BraTS 2021 label protocol.
Uses correct BraTS 2024 Post-Treatment label mapping: ET=label 3, NETC=label 1, SNFH=label 2. RC (label 4) excluded from evaluation per challenge spec.

**Phase 2 constraints applied:**
- `safe_loader_iter` for corrupt NIfTI files
- `CacheDataset(cache_rate=0.05)` for Train, plain `Dataset` for Val
- No `PersistentDataset` (kills Kaggle disk)
- `.nii_gz` symlink trick for Kaggle auto-extraction prevention

In [ ]:
from pathlib import Path
import os, warnings
warnings.filterwarnings('ignore')

MODEL_NAME  = 'swinunetr'
PATCH       = (128, 128, 128)
REGIONS     = ['WT', 'TC', 'ET']
OUTPUT_ROOT = Path('/kaggle/working/phase3_swinunetr')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Model: {MODEL_NAME} | Patch: {PATCH} | Regions: {REGIONS}')


In [ ]:
import subprocess, sys, json, time, math, shutil, gc, random
import numpy as np
import torch
import torch.nn.functional as F

# Install dependencies
for pkg in ['monai[all]', 'einops']:
    try:
        __import__(pkg.split('[')[0])
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import monai
import monai.transforms as T
from monai.data import Dataset, CacheDataset, DataLoader
from monai.losses import DiceLoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference
from monai.utils import set_determinism
from monai.transforms import MapTransform, Activations, AsDiscrete
from monai.networks.nets import SwinUNETR

set_determinism(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'MONAI {monai.__version__} | PyTorch {torch.__version__} | Device: {device}')
if torch.cuda.is_available():
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {total_mem:.1f} GB')
usage = shutil.disk_usage('/kaggle/working')
print(f'Disk free: {usage.free/1e9:.1f} GB')


In [ ]:
# BraTS 2024 Post-Treatment ground truth labels:
#   0 = Background
#   1 = NETC (Non-Enhancing Tumor Core — necrosis/cysts)
#   2 = SNFH (Surrounding Non-enhancing FLAIR Hyperintensity — edema)
#   3 = ET   (Enhancing Tissue — active tumor)
#   4 = RC   (Resection Cavity — fluid/blood/air)
#
# Evaluation sub-regions (from BraTS 2024 challenge spec):
#   ET = label 3
#   TC = ET + NETC       = labels 3 + 1
#   WT = ET + SNFH + NETC = labels 3 + 2 + 1
#   RC is NOT part of WT or TC!
#
# 3-channel output: [WT, TC, ET]

class ConvertToMultiChannelBrats2024(MapTransform):
    """Correct BraTS 2024 Post-Treatment label mapping."""
    def __call__(self, data):
        d = dict(data)
        for key in self.key_iterator(d):
            img = d[key]
            if img.ndim == 4 and img.shape[0] == 1:
                img = img.squeeze(0)
            result = [
                (img==1)|(img==2)|(img==3),  # WT = NETC+SNFH+ET (no RC)
                (img==1)|(img==3),           # TC = NETC+ET
                img==3,                      # ET = Enhancing Tissue
            ]
            d[key] = (torch.stack(result, 0).float()
                      if isinstance(img, torch.Tensor)
                      else np.stack(result, 0).astype(np.float32))
        return d

print('BraTS 2024 Label mapping: WT=1+2+3 | TC=1+3 | ET=3')
print('  1=NETC  2=SNFH  3=ET  4=RC(excluded)')


In [ ]:
# ═══════════ Data Discovery (identical to Phase 2) ═══════════
import nibabel as nib

SYMLINK_DIR = Path('/kaggle/working/nifti_links')

def setup_nii_gz_symlinks(data_dir):
    count = 0
    for nii_gz in Path(data_dir).rglob('*.nii_gz'):
        real_name = nii_gz.name.replace('.nii_gz', '.nii.gz')
        link = SYMLINK_DIR / nii_gz.parent.name / real_name
        link.parent.mkdir(parents=True, exist_ok=True)
        if not link.exists():
            os.symlink(str(nii_gz), str(link))
            count += 1
    return count

DATA_ROOT = Path('/kaggle/input')
for ds_dir in DATA_ROOT.iterdir():
    if not ds_dir.is_dir(): continue
    if list(ds_dir.rglob('*.nii_gz')):
        n = setup_nii_gz_symlinks(ds_dir)
        if n: print(f'  Created {n} symlinks in {ds_dir.name}')

NIFTI_ROOT = None
for search_root in [SYMLINK_DIR, DATA_ROOT]:
    if not search_root.exists(): continue
    for c in search_root.rglob('BraTS-GLI-*'):
        if c.is_dir():
            NIFTI_ROOT = c.parent
            break
    if NIFTI_ROOT: break

if NIFTI_ROOT is None:
    raise RuntimeError('No BraTS-GLI-* folders found — check dataset attachments')
print(f'NIFTI_ROOT: {NIFTI_ROOT}')

# Read split metadata
split_map, scan_meta = {}, {}
for f in DATA_ROOT.rglob('scan_index.json'):
    si = json.load(open(f))
    for s in si.get('training_scans', []):
        pid, sid = s['patient_id'], s['scan_id']
        split = s.get('split')
        if split: split_map[pid] = split
        scan_meta[sid] = {'patient_id': pid, 'timepoint': s.get('timepoint','100'), 'split': split}
    print(f'  Split metadata: {len(split_map)} patients from scan_index.json')
    break

all_dirs = sorted([d for d in NIFTI_ROOT.iterdir() if d.is_dir() and 'BraTS-GLI' in d.name])
training_scans = []
for d in all_dirs:
    files = {m: list(d.glob(f'*-{m}*')) for m in ['t1n','t1c','t2w','t2f']}
    seg   = list(d.glob('*-seg*'))
    if not (all(files[m] for m in files) and seg): continue
    name = d.name; pid = name.rsplit('-',1)[0]; tp = name.rsplit('-',1)[1] if '-' in name else '100'
    split = scan_meta.get(name, {}).get('split') or split_map.get(pid)
    training_scans.append({
        'scan_id': name, 'patient_id': pid, 'timepoint': tp,
        't1n': str(files['t1n'][0]), 't1c': str(files['t1c'][0]),
        't2w': str(files['t2w'][0]), 't2f': str(files['t2f'][0]),
        'seg': str(seg[0]), 'split': split,
    })
print(f'Total scans: {len(training_scans)}')

if any(s['split'] for s in training_scans):
    train_scans = [s for s in training_scans if s.get('split')=='train']
    val_scans   = [s for s in training_scans if s.get('split')=='val']
    kt = {s['patient_id'] for s in train_scans}; kv = {s['patient_id'] for s in val_scans}
    for s in training_scans:
        if s.get('split'): continue
        if s['patient_id'] in kt: train_scans.append(s)
        elif s['patient_id'] in kv: val_scans.append(s)
        else: train_scans.append(s)
else:
    from collections import defaultdict
    pts = defaultdict(list)
    for s in training_scans: pts[s['patient_id']].append(s)
    pids = sorted(pts.keys()); n80 = int(0.8*len(pids))
    tp_set = set(pids[:n80]); vp_set = set(pids[n80:])
    train_scans = [s for s in training_scans if s['patient_id'] in tp_set]
    val_scans   = [s for s in training_scans if s['patient_id'] in vp_set]

print(f'Train: {len(train_scans)} scans | Val: {len(val_scans)} scans')


In [ ]:
# ═══════════ Transforms ═══════════
patch = list(PATCH)
train_transforms = T.Compose([
    T.LoadImaged(keys=['image','label']),
    T.EnsureChannelFirstd(keys=['image','label']),
    T.EnsureTyped(keys=['image','label']),
    T.Orientationd(keys=['image','label'], axcodes='RAS'),
    T.CropForegroundd(keys=['image','label'], source_key='image', allow_smaller=True),
    T.NormalizeIntensityd(keys='image', nonzero=True, channel_wise=True),
    ConvertToMultiChannelBrats2024(keys=['label']),
    T.SpatialPadd(keys=['image','label'], spatial_size=patch),
    T.RandSpatialCropd(keys=['image','label'], roi_size=patch, random_size=False),
    T.RandFlipd(keys=['image','label'], spatial_axis=[0], prob=0.5),
    T.RandFlipd(keys=['image','label'], spatial_axis=[1], prob=0.5),
    T.RandFlipd(keys=['image','label'], spatial_axis=[2], prob=0.5),
    T.RandScaleIntensityd(keys='image', factors=0.1, prob=1.0),
    T.RandShiftIntensityd(keys='image', offsets=0.1, prob=1.0),
    T.EnsureTyped(keys=['image','label'], dtype=torch.float32),
])
val_transforms = T.Compose([
    T.LoadImaged(keys=['image','label']),
    T.EnsureChannelFirstd(keys=['image','label']),
    T.EnsureTyped(keys=['image','label']),
    T.Orientationd(keys=['image','label'], axcodes='RAS'),
    T.CropForegroundd(keys=['image','label'], source_key='image', allow_smaller=True),
    T.NormalizeIntensityd(keys='image', nonzero=True, channel_wise=True),
    ConvertToMultiChannelBrats2024(keys=['label']),
    T.EnsureTyped(keys=['image','label'], dtype=torch.float32),
])
print('Transforms ready (3-channel: WT/TC/ET, BraTS 2021 strict)')


In [ ]:
# ═══════════ Validate scans (gzip check from Phase 2) ═══════════
def is_valid_gzip(path):
    try:
        with open(str(path), 'rb') as f:
            h = f.read(10)
            if len(h) < 10 or h[0] != 31 or h[1] != 139: return False
            f.seek(0, 2)
            return f.tell() > 1024
    except Exception: return False

def validate_scan(s):
    try:
        for key in ['t1n','t1c','t2w','t2f','seg']:
            if not is_valid_gzip(s[key]): return False
            _ = nib.load(s[key]).shape
        return True
    except Exception: return False

def build_dicts(scan_list):
    dicts, bad = [], []
    for s in scan_list:
        if not validate_scan(s):
            bad.append(s['scan_id']); continue
        dicts.append({
            'image': [s['t1n'],s['t1c'],s['t2w'],s['t2f']],
            'label': s['seg'],
            'patient_id': s['patient_id'],
            'timepoint':  s['timepoint'],
        })
    if bad: print(f'  Skipped {len(bad)} corrupted: {bad[:3]}{"..." if len(bad)>3 else ""}')
    return dicts

print('Validating scans (gzip header check)...')
train_dicts = build_dicts(train_scans)
val_dicts   = build_dicts(val_scans)
print(f'Train dicts: {len(train_dicts)} | Val dicts: {len(val_dicts)}')


In [ ]:
# ═══════════ SwinUNETR Model + Pretrained Weights ═══════════
print('='*55)
print('  Loading SwinUNETR (BraTS 2021 Fold 1 — Dice 0.9059)')
print('='*55)

# MONAI 1.5.x: img_size is positional-only — pass PATCH directly
# Works across MONAI 0.9, 1.0, 1.3, 1.4, 1.5 without keyword errors
model = SwinUNETR(
        in_channels=4,
        out_channels=3,
        feature_size=48,
        use_checkpoint=True,
    )
print(f'  SwinUNETR created: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params')

# Find pretrained weights on Kaggle
weight_path = None
for p in Path('/kaggle/input').rglob('swinunetr_fold1.pt'):
    weight_path = p; break
if weight_path is None:
    for p in Path('/kaggle/input').rglob('model.pt'):
        if 'fold1' in str(p).lower() or 'swin' in str(p).lower():
            weight_path = p; break

if weight_path:
    print(f'  Found weights: {weight_path}')
    try:
        ckpt = torch.load(weight_path, map_location='cpu', weights_only=False)
    except TypeError:
        ckpt = torch.load(weight_path, map_location='cpu')
    # Official BraTS 2021 checkpoint key is 'state_dict'
    state = ckpt.get('state_dict', ckpt)
    own   = model.state_dict()
    compat = {k: v for k, v in state.items()
              if k in own and own[k].shape == v.shape}
    model.load_state_dict({**own, **compat}, strict=False)
    print(f'  Pretrained: {len(compat)}/{len(own)} layers loaded')
    if len(compat) < len(own) // 2:
        print('  WARNING: <50% layers matched — check weight file')
else:
    print('  WARNING: No pretrained weights found — training from scratch')
    print('  Expected: /kaggle/input/*/swinunetr_fold1.pt')

# Freeze patch embedding + first 2 Swin blocks for stability
frozen = 0
for name, param in model.named_parameters():
    if any(k in name for k in ['patch_embed', 'layers1', 'layers2',
                               'swinViT.patch_embed', 'swinViT.layers1',
                               'swinViT.layers2']):
        param.requires_grad = False
        frozen += 1

n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'  Total: {n_total/1e6:.1f}M | Trainable: {n_train/1e6:.1f}M | Frozen params: {frozen}')
model = model.to(device)
print(f'  Model on {device}')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# OPTION B — TEMPORAL CONTRASTIVE LOSS (v4)
# ═══════════════════════════════════════════════════════════════
# L_total = L_dice + λ_triplet × L_triplet
#
# Triplet mining per epoch:
#   Anchor  : patient_A, visit_t1
#   Positive: patient_A, visit_t2  (same patient, any other timepoint)
#   Negative: patient_B, any visit (different patient)
#
# Implementation strategy (batch_size=1 constraint):
#   - Each epoch pass: accumulate embeddings + (pid, tp) labels in RAM
#   - After every ACCUMULATE_STEPS forward passes: mine hard triplets
#     from the buffer, compute triplet loss, backprop through them
#   - This simulates a "virtual batch" of multi-patient embeddings
#   - No change to data loader or transforms

import random
from torch.nn import TripletMarginLoss

# ── Lightweight embedding head hooked onto encoder ──
# Hooks layers3[0] (Option A layer), applies ROI-aware global pool → 256-D projection
class TripletEmbeddingHead(torch.nn.Module):
    """Projects encoder features to a compact L2-normalised space for triplet loss.
    Uses LayerNorm (not BatchNorm1d) — BatchNorm1d fails with batch_size=1 (variance=0 → NaN).
    """
    def __init__(self, in_channels=384, proj_dim=256):
        super().__init__()
        self.pool = torch.nn.AdaptiveAvgPool3d(1)
        self.proj = torch.nn.Sequential(
            torch.nn.Flatten(),                        # (B,C,1,1,1)→(B,C)
            torch.nn.Linear(in_channels, proj_dim),
            torch.nn.LayerNorm(proj_dim),              # ← works with B=1 (not BatchNorm1d!)
            torch.nn.ReLU(inplace=True),
            torch.nn.Linear(proj_dim, proj_dim),
        )
    def forward(self, feat):
        # feat: (B, C, H, W, D) — e.g. (1, 384, 8, 8, 8) from layers3
        # Handle both spatial (5D) and sequence (3D) formats
        if feat.dim() == 3:                            # (B, tokens, C) — rearrange
            feat = feat.permute(0, 2, 1).unsqueeze(-1).unsqueeze(-1)  # (B,C,tokens,1,1)
        x = self.pool(feat)                            # (B, C, 1, 1, 1)
        x = self.proj(x)                               # (B, proj_dim)
        return torch.nn.functional.normalize(x, dim=1)  # L2-normalise


def mine_hard_triplets(embs, pids, margin=0.3, max_triplets=64):
    """
    Semi-hard negative mining from a buffer of embeddings.
    Returns (anchors, positives, negatives) tensor triplets.
    embs: (N, D) tensor
    pids: list of N patient-id strings
    """
    pid_arr = np.array(pids)
    triplets_a, triplets_p, triplets_n = [], [], []
    unique_pids = list(set(pids))
    random.shuffle(unique_pids)

    for pid in unique_pids:
        pos_idx = np.where(pid_arr == pid)[0]
        neg_idx = np.where(pid_arr != pid)[0]
        if len(pos_idx) < 2 or len(neg_idx) == 0:
            continue
        # All anchor-positive pairs for this patient
        for ai in pos_idx:
            for pi in pos_idx:
                if ai == pi: continue
                a_emb = embs[ai]   # (D,)
                p_emb = embs[pi]
                d_ap  = float(torch.dist(a_emb, p_emb))

                # Semi-hard negative: d_an > d_ap but d_an < d_ap + margin
                candidates = []
                for ni in neg_idx:
                    d_an = float(torch.dist(a_emb, embs[ni]))
                    if d_an > d_ap:  # semi-hard
                        candidates.append((ni, d_an))
                if not candidates:
                    # Fallback: hardest available negative
                    candidates = [(ni, float(torch.dist(a_emb, embs[ni]))) for ni in neg_idx]

                # Pick candidate closest to the anchor (hardest negative)
                best_ni = min(candidates, key=lambda x: x[1])[0]
                triplets_a.append(ai)
                triplets_p.append(pi)
                triplets_n.append(best_ni)

                if len(triplets_a) >= max_triplets:
                    break
            if len(triplets_a) >= max_triplets:
                break
        if len(triplets_a) >= max_triplets:
            break

    if not triplets_a:
        return None, None, None
    # Return indices (not tensors) — caller indexes stacked WITH grad
    return triplets_a, triplets_p, triplets_n


def train_model_v4(model,
     lr=5e-5, 
     epochs=10, 
     patience=6, 
     val_interval=2,
     lambda_triplet=0.3, 
     margin=0.3, 
     accumulate_steps=32):
    """
    Fine-tune with combined Dice + Temporal Triplet loss (Option B).
    
    Resumes from BEST checkpoint (frozen train → only additional triplet epochs).
    Output: swinunetr_best_v4.pth  (updated checkpoint)
    
    Args:
        lr              : Learning rate (lower than v1 — already converged on Dice)
        epochs          : Triplet fine-tuning epochs (10 is enough)
        lambda_triplet  : Weight of triplet loss (start 0.3, can tune)
        margin          : Triplet margin (0.3 standard for normalised embeddings)
        accumulate_steps: Number of scans buffered before triplet step
    """
    # ── Load best Dice checkpoint ──
    if not BEST_PATH.exists():
        print("  ERROR: No BEST checkpoint found — run base training first (Phase3_A1)")
        return model
    ck = torch.load(BEST_PATH, map_location=device)
    model.load_state_dict(ck['model'])
    start_dice = ck.get('best_dice', 0)
    print(f"  Loaded best checkpoint: epoch={ck.get('epoch','?')} Dice={start_dice:.4f}")

    # ── Build embedding head (hooks layers3) ──
    C_hook = 384  # layers3 expected channels (Option A)
    emb_head = TripletEmbeddingHead(in_channels=C_hook, proj_dim=256).to(device)

    # ── Hook: capture layers3 output during forward passes ──
    _buf = {}
    def _hook_fn(m, inp, out):
        feat = out[-1] if isinstance(out, (list, tuple)) else out
        _buf['feat'] = feat   # keep grad for backward

    sv = model.swinViT
    target_layer = 'layers3' if hasattr(sv, 'layers3') else 'layers2'
    C_hook = 384 if target_layer == 'layers3' else 192
    emb_head = TripletEmbeddingHead(in_channels=C_hook, proj_dim=256).to(device)

    layer_list = getattr(sv, target_layer)
    hook_target = layer_list[0] if hasattr(layer_list, '__getitem__') else layer_list
    hk = hook_target.register_forward_hook(_hook_fn)
    print(f"  Triplet hook: swinViT.{target_layer}[0] (C={C_hook})")

    # ── Optimiser: train both encoder + projection head ──
    optimizer = torch.optim.AdamW(
        list(filter(lambda p: p.requires_grad, model.parameters())) +
        list(emb_head.parameters()),
        lr=lr, weight_decay=1e-5
    )
    triplet_fn = TripletMarginLoss(margin=margin, p=2, reduction='mean')
    loss_fn    = DiceLoss(to_onehot_y=False, sigmoid=True, smooth_nr=0, smooth_dr=1e-5)
    dice_metric = DiceMetric(include_background=True, reduction='mean_batch')
    scaler = GradScaler()

    BEST_V4_PATH = CKPT_DIR / 'swinunetr_best_v4.pth'
    best_dice_v4 = start_dice

    # ── Build dataloaders (train_loader may not exist in global scope if training was skipped) ──
    from monai.data import CacheDataset
    _train_ds = CacheDataset(train_dicts, train_transforms, cache_rate=0.05, num_workers=2)
    _val_ds   = CacheDataset(val_dicts,   val_transforms,  cache_rate=0.05, num_workers=2)
    train_loader = DataLoader(_train_ds, batch_size=1, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(_val_ds,   batch_size=1, shuffle=False, num_workers=2, pin_memory=True)
    print(f"  Train: {len(train_loader)} batches | Val: {len(val_loader)} batches")

    print(f"  Training: {epochs} epochs | lr={lr} | λ_triplet={lambda_triplet} | margin={margin}")
    print(f"  Accumulate: {accumulate_steps} scans per triplet step")
    t0 = time.time()

    for ep in range(epochs):
        model.train(); emb_head.train()
        ep_dice_loss = ep_triplet_loss = 0.0
        n_ok = n_bad = n_triplet_steps = 0

        # Buffers for triplet mining
        buf_embs = []   # list of (proj_dim,) tensors WITH grad
        buf_pids = []   # patient-id strings

        for batch in safe_loader_iter(train_loader):
            try:
                imgs = batch['image'].to(device)
                lbls = batch['label'].to(device)
                pid  = batch['patient_id'][0] if 'patient_id' in batch else f"unk_{n_ok}"

                optimizer.zero_grad()
                _buf.clear()

                # ── Forward: Dice loss ──
                with autocast():
                    logits = model(imgs)
                    dice_loss = loss_fn(logits, lbls)

                # ── Extract embedding from hooked layer ──
                feat = _buf.get('feat', None)
                if feat is not None:
                    with autocast():
                        proj = emb_head(feat)   # (1, 256)
                    # Store DETACHED proj for mining + the last live proj separately
                    buf_embs.append(proj[0].detach())    # detached — safe after backward
                    buf_pids.append(pid)

                is_accumulate = len(buf_embs) >= accumulate_steps and len(set(buf_pids)) >= 4

                if is_accumulate:
                    # ── TRIPLET STEP: mine from detached buffer ──
                    det_stacked = torch.stack(buf_embs, dim=0)  # (N, 256) detached
                    a_idx, p_idx, n_idx = mine_hard_triplets(det_stacked, buf_pids,
                                                  margin=margin, max_triplets=64)
                    if a_idx is not None and feat is not None:
                        # Re-project CURRENT scan's feat (still has grad) for triplet
                        # This is the only scan with a live computation graph
                        with autocast():
                            live_proj = emb_head(feat)[0]   # (256,) WITH grad
                        # Build triplet targets from buffer (detached = no grad for others)
                        # But current scan's embedding has grad → encoder + emb_head learn
                        buf_idx = len(buf_embs) - 1  # last = current scan
                        # Replace current scan's detached entry with live version
                        live_buf = list(buf_embs)  # copy
                        live_buf[buf_idx] = live_proj
                        live_stacked = torch.stack(live_buf, dim=0)
                        with autocast():
                            t_loss = triplet_fn(
                                live_stacked[a_idx],
                                live_stacked[p_idx],
                                live_stacked[n_idx]
                            )
                        total_loss = dice_loss + lambda_triplet * t_loss
                        scaler.scale(total_loss).backward()
                        scaler.step(optimizer); scaler.update()
                        ep_dice_loss    += dice_loss.item()
                        ep_triplet_loss += t_loss.item()
                        n_triplet_steps += 1
                    else:
                        # No valid triplets found — just do dice
                        scaler.scale(dice_loss).backward()
                        scaler.step(optimizer); scaler.update()
                        ep_dice_loss += dice_loss.item()
                    buf_embs.clear(); buf_pids.clear()
                else:
                    # ── ACCUMULATING: pure Dice backward ──
                    scaler.scale(dice_loss).backward()
                    scaler.step(optimizer); scaler.update()
                    ep_dice_loss += dice_loss.item()

                n_ok += 1
                optimizer.zero_grad()

            except Exception as e:
                n_bad += 1
                if n_bad <= 3:
                    print(f"  [WARN ep={ep} batch={n_ok+n_bad}] {type(e).__name__}: {str(e)[:120]}")

        avg_dice_l  = ep_dice_loss  / max(n_ok, 1)
        avg_triple_l = ep_triplet_loss / max(n_triplet_steps, 1)

        # ── Validation ──
        if (ep+1) % val_interval == 0 or ep == epochs-1:
            model.eval(); dice_metric.reset()
            with torch.no_grad():
                for vb in safe_loader_iter(val_loader):
                    try:
                        vo = sliding_window_inference(vb['image'].to(device), PATCH, 4, model, overlap=0.5)
                        dice_metric((torch.sigmoid(vo)>0.5).float(), vb['label'].to(device))
                    except Exception:
                        pass
            dv = dice_metric.aggregate(); md = dv.mean().item()
            pr = [round(dv[i].item(), 4) for i in range(3)]
            tag = ' ⭐ NEW BEST' if md > best_dice_v4 else ''
            print(f"Ep {ep:2d} | L_dice={avg_dice_l:.4f} L_tri={avg_triple_l:.4f} "
                  f"| Dice={md:.4f} WT={pr[0]:.3f} TC={pr[1]:.3f} ET={pr[2]:.3f} "
                  f"| tri_steps={n_triplet_steps} | {(time.time()-t0)/60:.1f}m{tag}")
            if md > best_dice_v4:
                best_dice_v4 = md
                torch.save({'model': model.state_dict(),
                            'emb_head': emb_head.state_dict(),
                            'epoch': ep, 'best_dice': best_dice_v4,
                            'lambda_triplet': lambda_triplet,
                            'target_layer': target_layer}, BEST_V4_PATH)
        else:
            print(f"Ep {ep:2d} | L_dice={avg_dice_l:.4f} L_tri={avg_triple_l:.4f} "
                  f"| tri_steps={n_triplet_steps} | ok={n_ok} bad={n_bad} | {(time.time()-t0)/60:.1f}m")

    hk.remove()
    print(f"\n  Option B complete. Best v4 Dice={best_dice_v4:.4f}")
    print(f"  Checkpoint: {BEST_V4_PATH}")
    return model, emb_head, best_dice_v4


print("  Option B (Temporal Contrastive) — ready to run")
print("  Call: model, emb_head, bdice = train_model_v4(model)")
print("  Then: extract_embeddings_v4(model, emb_head)  ← next cell")


In [ ]:
from torch.cuda.amp import GradScaler, autocast

CKPT_DIR    = OUTPUT_ROOT / 'checkpoints'; CKPT_DIR.mkdir(exist_ok=True)
BEST_PATH   = CKPT_DIR / 'swinunetr_best.pth'
LATEST_PATH = CKPT_DIR / 'swinunetr_latest.pth'

# Recover checkpoints from previous notebook output (attached as input)
if not BEST_PATH.exists() or not LATEST_PATH.exists():
    for src_f in sorted(Path('/kaggle/input').rglob('swinunetr_best.pth')):
        if not BEST_PATH.exists():
            shutil.copy2(src_f, BEST_PATH)
            print(f'  Recovered BEST checkpoint from {src_f}')
        break
    for src_f in sorted(Path('/kaggle/input').rglob('swinunetr_latest.pth')):
        if not LATEST_PATH.exists():
            shutil.copy2(src_f, LATEST_PATH)
            print(f'  Recovered LATEST checkpoint from {src_f}')
        break
    if BEST_PATH.exists():
        print(f'  \u2705 Checkpoints recovered — training will be skipped')

def get_lr(ep, total, base=1e-4):
    warm = 5
    if ep < warm: return base * (ep+1) / warm
    return base * 0.5 * (1 + math.cos(math.pi * (ep-warm) / max(total-warm, 1)))

def safe_loader_iter(loader):
    it = iter(loader)
    SKIP = ['EOFError','gzip','Compressed file','end-of-stream','worker','corrupt','truncat']
    while True:
        try:
            yield next(it)
        except StopIteration:
            return
        except Exception as e:
            if any(k in str(e) for k in SKIP):
                continue
            raise

def train_model(model, lr=1e-4, epochs=28, patience=12, val_interval=4):
    start_ep, best_dice, mlog = 0, 0.0, {'dice':[],'per_region':[],'loss':[]}
    if LATEST_PATH.exists():
        lc = torch.load(LATEST_PATH, map_location='cpu')
        model.load_state_dict(lc['model'])
        start_ep  = lc.get('epoch',0) + 1
        best_dice = lc.get('best_dice', 0)
        mlog      = lc.get('metrics', mlog)
        print(f'Resumed from epoch {start_ep-1}, best_dice={best_dice:.4f}')
        if start_ep >= epochs:
            return model, best_dice, mlog

    loss_fn     = DiceLoss(to_onehot_y=False, sigmoid=True, smooth_nr=0, smooth_dr=1e-5)
    optimizer   = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                     lr=lr, weight_decay=1e-5)
    scaler      = GradScaler()
    dice_metric = DiceMetric(include_background=True, reduction='mean_batch')
    no_improve  = 0
    t0          = time.time()

    try:
        train_ds = CacheDataset(train_dicts, train_transforms, cache_rate=0.05, num_workers=4)
        print('Using CacheDataset (5% RAM cache)')
    except Exception:
        train_ds = Dataset(train_dicts, train_transforms)
        print('Using plain Dataset')
    val_ds = Dataset(val_dicts, val_transforms)

    train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,  num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False, num_workers=4, pin_memory=True)
    print(f'Train: {len(train_loader)} batches | Val: {len(val_loader)} batches')

    for ep in range(start_ep, epochs):
        model.train()
        cur_lr = get_lr(ep, epochs, lr)
        for pg in optimizer.param_groups: pg['lr'] = cur_lr

        ep_loss, n_ok, n_bad = 0.0, 0, 0
        for batch in safe_loader_iter(train_loader):
            try:
                imgs = batch['image'].to(device)
                lbls = batch['label'].to(device)
                optimizer.zero_grad()
                with autocast():
                    loss = loss_fn(model(imgs), lbls)
                scaler.scale(loss).backward()
                scaler.step(optimizer); scaler.update()
                ep_loss += loss.item(); n_ok += 1
            except Exception:
                n_bad += 1
        avg_loss = ep_loss / max(n_ok, 1)
        bad_str  = f' | skipped {n_bad}' if n_bad else ''
        mlog['loss'].append(avg_loss)

        if (ep+1) % val_interval == 0 or ep == epochs-1:
            model.eval(); dice_metric.reset()
            with torch.no_grad():
                for vb in safe_loader_iter(val_loader):
                    try:
                        vo = sliding_window_inference(vb['image'].to(device), PATCH, 4, model, overlap=0.5)
                        dice_metric((torch.sigmoid(vo)>0.5).float(), vb['label'].to(device))
                    except Exception:
                        pass
            dv = dice_metric.aggregate(); md = dv.mean().item()
            pr = [round(dv[i].item(),4) for i in range(3)]
            mlog['dice'].append(md); mlog['per_region'].append(pr)
            tag = ' \u2b50 NEW BEST' if md > best_dice else ''
            print(f'Ep {ep:3d} | L={avg_loss:.4f} | Dice={md:.4f} WT={pr[0]:.3f} TC={pr[1]:.3f} ET={pr[2]:.3f} | {(time.time()-t0)/60:.1f}m{tag}{bad_str}')
            if md > best_dice:
                best_dice = md; no_improve = 0
                torch.save({'model': model.state_dict(), 'epoch': ep, 'best_dice': best_dice}, BEST_PATH)
            else:
                no_improve += val_interval
        else:
            print(f'Ep {ep:3d} | L={avg_loss:.4f} | LR={cur_lr:.2e} | {(time.time()-t0)/60:.1f}m{bad_str}')

        torch.save({'model': model.state_dict(), 'epoch': ep,
                    'best_dice': best_dice, 'metrics': mlog}, LATEST_PATH)
        if no_improve >= patience:
            print(f'Early stop at ep {ep}'); break

    print(f'Done. Best Mean Dice = {best_dice:.4f}')
    return model, best_dice, mlog

print('Fine-tuning SwinUNETR on BraTS 2024...')
model, best_dice, metrics = train_model(model, lr=1e-4, epochs=28, patience=12, val_interval=4)


In [ ]:
# ── RUN OPTION B ──
# Resumes from BEST checkpoint → adds temporal contrastive loss
# GPU: YES (same as base training ~2-3h for 10 epochs on T4)
# Skip this if you only want Option A results

SKIP_OPTION_B = False   # set True to skip and keep base model

if not SKIP_OPTION_B:
    print("="*60)
    print("  OPTION B — Temporal Contrastive Fine-tuning")
    print("="*60)
    model, emb_head, best_dice_v4 = train_model_v4(
        model,
        lr=5e-5,            # lower lr — model already converged on Dice
        epochs=20,          # enough for contrastive to re-organise embeddings
        patience=6,
        val_interval=2,
        lambda_triplet=0.3, # 0.3 = balanced; increase if M6 still low
        margin=0.3,         # standard for L2-normalised embeddings
        accumulate_steps=32 # 32 scans = ~4-8 patients per triplet step
    )
else:
    print("  SKIP_OPTION_B=True → using base model (Option A only)")
    emb_head = None


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

fig_dir = OUTPUT_ROOT / 'figures'; fig_dir.mkdir(exist_ok=True)
REGION_COLORS = {'WT': ('red','lightcoral'), 'TC': ('green','lightgreen'), 'ET': ('blue','lightskyblue')}

# ── Load BEST checkpoint before visualization ──────────────────────
# train_model returns with LATEST weights; reload BEST for fair eval
if BEST_PATH.exists():
    best_ckpt = torch.load(BEST_PATH, map_location='cpu')
    model.load_state_dict(best_ckpt['model'])
    print(f'  Loaded BEST checkpoint → epoch {best_ckpt["epoch"]} | Dice {best_ckpt["best_dice"]:.4f}')
else:
    print('  BEST checkpoint not found — using current model weights')

# Flush optimizer/scaler VRAM before inference
for _var in ['optimizer', 'scaler', 'train_ds', 'train_loader']:
    if _var in dir(): exec(f'del {_var}')
gc.collect(); torch.cuda.empty_cache()
print(f'  VRAM free after flush: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.1f} GB')

def visualize_3d_predictions(model, n_samples=5):
    """3D voxel-scatter GT vs Prediction. Random patients, sw_batch_size=1, overlap=0.25."""
    model.eval()
    vis_sample = random.sample(val_dicts, min(n_samples, len(val_dicts)))
    print(f'  Visualising {len(vis_sample)} random val patients:')
    for s in vis_sample: print(f'    {s["patient_id"]} tp={s["timepoint"]}')
    vis_ds = Dataset(vis_sample, val_transforms)
    vis_loader = DataLoader(vis_ds, batch_size=1, shuffle=False, num_workers=0)
    dice_metric = DiceMetric(include_background=True, reduction='none')
    for i, batch in enumerate(vis_loader):
        try:
            gc.collect(); torch.cuda.empty_cache()
            vi = batch['image'].to(device); vl = batch['label'].to(device)
            with torch.no_grad():
                vo = sliding_window_inference(vi, PATCH, 1, model, overlap=0.25)
            pred_bin = (torch.sigmoid(vo) > 0.5).float()
            dice_metric.reset(); dice_metric(pred_bin, vl)
            dv = dice_metric.aggregate()[0].cpu().numpy()
            pid = batch.get('patient_id', ['?'])[0]
            # Handle NaN (empty GT region = complete resection)
            def fmt(v): return 'NaN(empty GT)' if np.isnan(v) else f'{v:.3f}'
            dstr = '  '.join(f'{r}={fmt(dv[j])}' for j,r in enumerate(REGIONS))
            mean_d = float(np.nanmean(dv))
            title_str = f'Sample {i} | {dstr}  Mean(non-NaN)={mean_d:.3f}  {pid}'
            pred_np = pred_bin.squeeze(0).cpu().numpy()
            gt_np = vl.squeeze(0).cpu().numpy()
            del vi, vl, vo, pred_bin; gc.collect(); torch.cuda.empty_cache()
            STEP = 3
            fig = plt.figure(figsize=(18, 7)); fig.suptitle(title_str, fontsize=10, y=1.01)
            for col, (arr, col_title) in enumerate([(gt_np, 'Ground Truth'), (pred_np, 'SwinUNETR Prediction')]):
                ax = fig.add_subplot(1, 2, col+1, projection='3d')
                ax.set_title(col_title, fontsize=11, pad=8)
                for r_idx, rname in enumerate(REGIONS):
                    mask = arr[r_idx]; coords = (mask > 0.5).nonzero()
                    if len(coords[0]) == 0: continue
                    xs, ys, zs = coords[0][::STEP], coords[1][::STEP], coords[2][::STEP]
                    color = REGION_COLORS[rname][0] if col==0 else REGION_COLORS[rname][1]
                    ax.scatter(xs, ys, zs, c=color, alpha=0.25, s=0.8, label=rname)
                ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
                ax.tick_params(labelsize=7); ax.legend(markerscale=6, loc='upper left', fontsize=8)
            plt.tight_layout()
            out_png = fig_dir / f'swinunetr_3d_sample{i}.png'
            plt.savefig(out_png, dpi=120, bbox_inches='tight'); plt.close()
            print(f'  Saved: {out_png.name} | {dstr}')
        except torch.cuda.OutOfMemoryError:
            gc.collect(); torch.cuda.empty_cache()
            print(f'  OOM sample {i} — skipping')
        except Exception as e:
            print(f'  Skipped {i}: {e}')

print('Generating 3D voxel-scatter visualizations (BEST checkpoint, 5 random patients)...')
visualize_3d_predictions(model, n_samples=5)


In [ ]:
def visualize_2d_overlays(model, n_samples=5):
    """2D slice overlay — 5 random patients, sw_batch_size=1, overlap=0.25."""
    model.eval()
    vis_sample = random.sample(val_dicts, min(n_samples, len(val_dicts)))
    vis_ds = Dataset(vis_sample, val_transforms)
    vis_loader = DataLoader(vis_ds, batch_size=1, shuffle=False, num_workers=0)
    for i, batch in enumerate(vis_loader):
        try:
            gc.collect(); torch.cuda.empty_cache()
            img = batch['image'].to(device); lbl = batch['label'].to(device)
            with torch.no_grad():
                vo = sliding_window_inference(img, PATCH, 1, model, overlap=0.25)
            pred_bin = (torch.sigmoid(vo) > 0.5).float().squeeze(0).cpu().numpy()
            t1c_np = img[0, 1].cpu().numpy()  # free GPU immediately
            del img, lbl, vo; gc.collect(); torch.cuda.empty_cache()
            wt_mask = pred_bin[0]
            if wt_mask.sum() == 0:
                print(f'  Sample {i}: no WT voxels — skip'); continue
            z_slice = int(np.argmax(wt_mask.sum(axis=(0,1))))
            t1c_slice = t1c_np[:, :, z_slice]
            edema     = np.logical_and(pred_bin[0,:,:,z_slice]>0, pred_bin[1,:,:,z_slice]==0)
            necrosis  = np.logical_and(pred_bin[1,:,:,z_slice]>0, pred_bin[2,:,:,z_slice]==0)
            enhancing = pred_bin[2,:,:,z_slice] > 0
            mask_rgb = np.zeros((*t1c_slice.shape, 3))
            mask_rgb[edema]     = [0,1,0]  # Green = SNFH
            mask_rgb[enhancing] = [0,0,1]  # Blue  = ET
            mask_rgb[necrosis]  = [1,0,0]  # Red   = NETC
            fig, axes = plt.subplots(1, 2, figsize=(14, 6))
            axes[0].set_title('Original T1c (z={})'.format(z_slice))
            axes[0].imshow(t1c_slice, cmap='gray'); axes[0].axis('off')
            axes[1].set_title('Overlay (G:SNFH, B:ET, R:NETC)')
            axes[1].imshow(t1c_slice, cmap='gray')
            axes[1].imshow(mask_rgb, alpha=0.45); axes[1].axis('off')
            plt.tight_layout()
            out_path = fig_dir / f'swinunetr_2d_sample{i}.png'
            plt.savefig(out_path, dpi=120, bbox_inches='tight'); plt.close()
            print(f'  Saved: {out_path.name}')
        except torch.cuda.OutOfMemoryError:
            gc.collect(); torch.cuda.empty_cache()
            print(f'  OOM sample {i} — skipping')
        except Exception as e:
            print(f'  Skip 2D sample {i}: {e}')

print('Generating 2D overlay visualizations (5 random patients)...')
visualize_2d_overlays(model, n_samples=5)
print(f'Figures saved to: {fig_dir}')


In [ ]:
# ═══════════ ViT Embedding Extraction v2 ════════════════════════
#
# Research Question:
#   Can Vision Transformer-based models capture meaningful
#   representations of tumor morphology and its EVOLUTION
#   from longitudinal medical imaging?
#
# v2 Design — ROI-Crop + Octant Spatial Pooling:
# ─────────────────────────────────────────────────────────────
# PROBLEM (v1): Global avg from layers4 (4×4×4) encodes brain
#   anatomy (99.7% identical across all scans), not tumor.
#
# FIX (v2): Crop feature maps to tumor ROI, then pool WITHIN
#   the tumor region. No brain anatomy noise.
#
# Pipeline:
#   1. Run full 128³ brain through SwinUNETR (preserving attention)
#   2. Hook layers3[0] → 8×8×8, C=384 (deep semantic features)
#   3. Find WT bounding box in 16³ feature space + 1-cell padding
#   4. Crop feature map to tumor ROI
#   5. Octant pool: adaptive_avg_pool3d to 2×2×2 → 8×384 = 3072-D
#   6. Mask-weighted pool: WT/TC/ET soft masks → 3×192 = 576-D
#   7. Volumetric morphology: 9-D explicit features
#
# TOTAL: 1536 + 576 + 9 = 2121-D per scan
#
# For complete resection (no tumor):
#   → Octant = zero vector (1536-D) = "nothing here"
#   → Region = zero vector (576-D)  = "absent"
#   → Vol: has_wt=0, log_wt=0, etc. = explicit "resected"
# ═══════════════════════════════════════════════════════════════

def _is_corrupt(exc):
    SKIP = ['EOFError','gzip','Compressed file','end-of-stream',
            'corrupt','truncat','LoadImaged','applying transform']
    e = exc
    while e is not None:
        if any(k in (type(e).__name__+' '+str(e)) for k in SKIP): return True
        e = e.__cause__ or e.__context__
    return False

def safe_emb_iter(loader):
    it, n_skip = iter(loader), 0
    while True:
        try: yield next(it)
        except StopIteration:
            if n_skip: print(f'  Skipped {n_skip} corrupt files total')
            return
        except Exception as e:
            if _is_corrupt(e): n_skip += 1; continue
            raise

def get_wt_bbox(lbl_feat, min_size=2):
    """Find WT bounding box in feature-space label map.
    lbl_feat: (3, h, w, d) soft labels at feature resolution.
    Returns: (z0,z1, y0,y1, x0,x1) with 1-cell padding, or None if empty.
    """
    wt = lbl_feat[0]  # WT channel
    # Threshold soft labels → nonzero mask
    mask = (wt > 0.01).nonzero(as_tuple=False)  # (N, 3)
    if len(mask) < 1:
        return None
    z_min, y_min, x_min = mask.min(dim=0).values.tolist()
    z_max, y_max, x_max = mask.max(dim=0).values.tolist()
    h, w, d = wt.shape
    # Add 1-cell padding, clamp to boundaries
    z0 = max(z_min - 1, 0);   z1 = min(z_max + 2, h)
    y0 = max(y_min - 1, 0);   y1 = min(y_max + 2, w)
    x0 = max(x_min - 1, 0);   x1 = min(x_max + 2, d)
    # Ensure minimum size of 2 per dim (for octant pooling)
    if z1 - z0 < min_size: z1 = min(z0 + min_size, h)
    if y1 - y0 < min_size: y1 = min(y0 + min_size, w)
    if x1 - x0 < min_size: x1 = min(x0 + min_size, d)
    return (z0, z1, y0, y1, x0, x1)

def extract_embeddings(model):
    model.eval()
    _feats = {}; hooks = []

    # ── Hook layers3[0]: 8×8×8, C=384 (deep semantic features) ──
    # layers2 (16³, C=192): mid-level, octant CV=0.087 (very homogeneous)
    # layers3 (8³, C=384): deep semantic, expected CV>>0.087, rank>>118
    sv = model.swinViT
    target_layer = 'layers3'
    C_feat = 384  # default for layers3; overridden to 192 if fallback
    # Fallback to layers2 if layers3 not available
    if not hasattr(sv, target_layer):
        target_layer = 'layers2'
        C_feat = 192
        print(f'  WARNING: layers3 not found, falling back to {target_layer}')
    if hasattr(sv, target_layer):
        layer_list = getattr(sv, target_layer)
        target = layer_list[0] if hasattr(layer_list,'__getitem__') and len(layer_list)>0 else layer_list
        def _hook(m, inp, out):
            feat = out[-1] if isinstance(out,(list,tuple)) else out
            _feats['feat'] = feat.detach()
        hooks.append(target.register_forward_hook(_hook))
        print(f'  Hook: swinViT.{target_layer}[0] ({type(target).__name__}) → C_feat={C_feat}')

    emb_dir = OUTPUT_ROOT / 'embeddings'; emb_dir.mkdir(exist_ok=True)
    embs, ids, tps = [], [], []
    spatial_tokens_list = []  # Phase 4/5: raw spatial features before pooling
    bboxes_list = []          # ROI bounding boxes
    all_dicts = train_dicts + val_dicts
    ds     = Dataset(all_dicts, val_transforms)
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0)
    total  = len(all_dicts)
    n_skip = 0; n_empty = 0; n_small_roi = 0
    t_start = time.time()

    # C_feat already set above (384 for layers3, 192 for layers2 fallback)
    OCT_DIM = 8 * C_feat
    REG_DIM = 3 * C_feat
    VOL_DIM = 9
    TOTAL_DIM = OCT_DIM + REG_DIM + VOL_DIM

    print(f'  Extraction v3 (layers3): {total} scans → ~{TOTAL_DIM}-D embeddings (exact C from hook)')
    print(f'  (octant={OCT_DIM} + region={REG_DIM} + vol={VOL_DIM})')
    print(f'  {"─"*60}')

    with torch.no_grad():
        for idx, batch in enumerate(safe_emb_iter(loader)):
            pid = batch['patient_id'][0]
            tp  = batch['timepoint'][0]
            try:
                img = batch['image'].to(device)
                lbl = batch['label'].to(device)  # (1, 3, H, W, D)
                img_p = F.interpolate(img, list(PATCH), mode='trilinear', align_corners=False)
                lbl_p = F.interpolate(lbl.float(), list(PATCH), mode='nearest')
                _feats.clear()
                _ = model(img_p)

                if 'feat' not in _feats:
                    print(f'  [{idx+1:4d}/{total}] WARN no hook → skip {pid}')
                    n_skip += 1; continue

                feat = _feats['feat']  # (1, C, 16, 16, 16)
                C = feat.shape[1]
                h, w, d = feat.shape[2], feat.shape[3], feat.shape[4]

                if idx == 0:
                    C_feat = C
                    OCT_DIM = 8 * C; REG_DIM = 3 * C
                    TOTAL_DIM = OCT_DIM + REG_DIM + VOL_DIM
                    print(f'  Feature map: {tuple(feat.shape)} → C={C}')
                    print(f'  Embedding: octant={OCT_DIM} + region={REG_DIM} + vol={VOL_DIM} = {TOTAL_DIM}-D')
                    print(f'  {"─"*60}')

                # Soft-downsample labels to feature resolution
                lbl_feat = F.adaptive_avg_pool3d(lbl_p, (h, w, d))  # (1, 3, 16, 16, 16)

                # ── Volumes at ORIGINAL resolution (for Component 3) ──
                wt_vol = float(lbl_p[0, 0].sum().item())
                tc_vol = float(lbl_p[0, 1].sum().item())
                et_vol = float(lbl_p[0, 2].sum().item())

                # ── Component 1: Octant Spatial Pooling (8 × C) ──
                feat_crop = None  # initialize so del is always safe
                bbox = get_wt_bbox(lbl_feat[0], min_size=2)

                if bbox is not None:
                    z0, z1, y0, y1, x0, x1 = bbox
                    feat_crop = feat[:, :, z0:z1, y0:y1, x0:x1]  # (1, C, dz, dy, dx)
                    # Octant pool: 2×2×2 adaptive avg → 8 spatial cells × C features
                    oct_pooled = F.adaptive_avg_pool3d(feat_crop, (2, 2, 2))  # (1, C, 2, 2, 2)
                    oct_vec = oct_pooled[0].reshape(C, 8).T.reshape(-1)       # (8C,)
                    # Mask-weighted pool: WT/TC/ET within ROI
                    lbl_crop = lbl_feat[:, :, z0:z1, y0:y1, x0:x1]  # (1, 3, dz, dy, dx)
                    feat_flat = feat_crop[0].reshape(C, -1)  # (C, N)
                    region_vecs = []
                    for ch in range(3):  # WT, TC, ET
                        mask = lbl_crop[0, ch].reshape(-1)  # (N,) soft
                        vol_soft = float(mask.sum().item())
                        if vol_soft > 0.01:
                            rvec = (feat_flat * mask.unsqueeze(0)).sum(1) / mask.sum()
                        else:
                            rvec = torch.zeros(C, device=device)
                        region_vecs.append(rvec)
                    roi_size = f'{z1-z0}×{y1-y0}×{x1-x0}'
                    # ── Capture spatial tokens BEFORE del feat_crop ──
                    sp_tok = feat_crop[0].reshape(C, -1).T.cpu().numpy()  # (N_tok, C)
                    bbox_entry = list(bbox)
                else:
                    # Complete resection — no tumor ROI
                    oct_vec = torch.zeros(8 * C, device=device)
                    region_vecs = [torch.zeros(C, device=device) for _ in range(3)]
                    roi_size = 'empty'
                    n_empty += 1
                    sp_tok = np.zeros((1, C), dtype=np.float32)  # explicit 'absent' token
                    bbox_entry = [0, 0, 0, 0, 0, 0]

                # ── Component 3: Volumetric morphology (9-D) ──
                log_wt = np.log1p(wt_vol)
                log_tc = np.log1p(tc_vol)
                log_et = np.log1p(et_vol)
                has_wt = 1.0 if wt_vol > 10 else 0.0
                has_tc = 1.0 if tc_vol > 10 else 0.0
                has_et = 1.0 if et_vol > 10 else 0.0
                tc_wt = tc_vol / (wt_vol + 1e-6)
                et_wt = et_vol / (wt_vol + 1e-6)
                et_tc = et_vol / (tc_vol + 1e-6)
                vol_feat = torch.tensor(
                    [log_wt, log_tc, log_et, has_wt, has_tc, has_et,
                     tc_wt, et_wt, et_tc], dtype=torch.float32)

                # ── Concatenate: [octant(8C) + region(3C) + vol(9)] ──
                emb = torch.cat([oct_vec.cpu()] + [v.cpu() for v in region_vecs]
                                + [vol_feat]).numpy()

                # cleanup GPU tensors (feat_crop may be None if no ROI)
                del img, lbl, img_p, lbl_p, feat
                if feat_crop is not None:
                    del feat_crop
                    feat_crop = None
                # Append pre-computed spatial tokens (captured before del above)
                spatial_tokens_list.append(sp_tok)
                bboxes_list.append(bbox_entry)

                embs.append(emb); ids.append(pid); tps.append(tp)

                # ── Progress ──
                if (idx+1) % 50 == 0 or idx == 0:
                    elapsed   = time.time() - t_start
                    rate      = (idx+1) / max(elapsed, 1e-6)
                    remaining = (total - idx - 1) / max(rate, 1e-6)
                    print(
                        f'  [{idx+1:4d}/{total}] {pid[:30]:<30}'
                        f'  tp={tp}'
                        f'  WT={wt_vol:6.0f}v TC={tc_vol:6.0f}v ET={et_vol:6.0f}v'
                        f'  ROI={roi_size}'
                        f'  | {rate:.1f}/s ETA {remaining/60:.1f}m'
                    )
                elif (idx+1) % 10 == 0:
                    elapsed   = time.time() - t_start
                    rate      = (idx+1) / max(elapsed, 1e-6)
                    remaining = (total - idx - 1) / max(rate, 1e-6)
                    print(f'  [{idx+1:4d}/{total}]  done={len(embs)} | {rate:.1f}/s ETA {remaining/60:.1f}m')

            except Exception as e:
                print(f'  [{idx+1:4d}/{total}] ERROR {pid}: {str(e)[:80]}')
                n_skip += 1; continue

    for hk in hooks:
        try: hk.remove()
        except: pass

    elapsed_total = time.time() - t_start
    print(f'  {"─"*60}')
    print(f'  Done: {len(embs)}/{total} in {elapsed_total/60:.1f} min')
    print(f'  Skipped: {n_skip} | Empty ROI (resected): {n_empty}')
    print(f'  Embedding dim: {TOTAL_DIM}  (octant={OCT_DIM} + region={REG_DIM} + vol={VOL_DIM})')

    if not embs: raise RuntimeError('No embeddings extracted.')
    arr = np.array(embs)
    out = emb_dir / 'vit_swinunetr_embeddings_v3.npz'  # layers3 deeper hook
    # ── Also save spatial tokens for Phase 4 (RadFM) + Phase 5 (TaDiff) ──
    # These are the ROI-cropped feature maps BEFORE pooling
    # RadFM Perceiver cross-attends over these spatial tokens
    # TaDiff uses them as conditioning in cross-attention UNet layers
    if spatial_tokens_list:
        # Pad to max token count
        max_tokens = max(t.shape[0] for t in spatial_tokens_list)
        C_tok = spatial_tokens_list[0].shape[1]
        padded = np.zeros((len(spatial_tokens_list), max_tokens, C_tok), dtype=np.float32)
        n_tokens = np.zeros(len(spatial_tokens_list), dtype=np.int32)
        for j, tok in enumerate(spatial_tokens_list):
            padded[j, :tok.shape[0], :] = tok
            n_tokens[j] = tok.shape[0]
        tok_out = emb_dir / 'vit_spatial_tokens_v3.npz'  # layers3 spatial tokens
        np.savez_compressed(tok_out,
            spatial_tokens=padded,      # (N, T_max, C) where C=384 for layers3
            token_counts=n_tokens,       # (N,)
            patient_ids=np.array(ids),
            timepoints=np.array(tps),
            bboxes=np.array(bboxes_list) # (N, 6)
        )
        print(f'  Spatial tokens: {tok_out}')
        print(f'    Shape: {padded.shape} ({padded.nbytes/1e6:.1f} MB)')
        print(f'    Max tokens: {max_tokens}  Mean: {n_tokens.mean():.0f}')
    
    np.savez_compressed(out, embeddings=arr,
                        patient_ids=np.array(ids),
                        timepoints=np.array(tps))
    print(f'  Saved: {out}')
    print(f'  Shape: {arr.shape} ({arr.nbytes/1e6:.1f} MB)')

    # ── Save tumor_volumes.csv for eval notebook (M1-M6 tests) ──
    import pandas as pd
    vol_rows = []
    for j, (pid, tp, emb) in enumerate(zip(ids, tps, embs)):
        v = emb[-9:]  # last 9 dims = volumetric features
        vol_rows.append({
            'patient_id':   pid,
            'timepoint':    tp,
            'wt_vol':       float(np.expm1(v[0])),  # inverse log1p
            'tc_vol':       float(np.expm1(v[1])),
            'et_vol':       float(np.expm1(v[2])),
            'has_wt':       float(v[3]),
            'has_tc':       float(v[4]),
            'has_et':       float(v[5]),
            'tc_wt_ratio':  float(v[6]),
            'et_wt_ratio':  float(v[7]),
            'et_tc_ratio':  float(v[8]),
        })
    vol_df  = pd.DataFrame(vol_rows)
    csv_out = emb_dir / 'tumor_volumes.csv'
    vol_df.to_csv(csv_out, index=False)
    print(f'  Volumes CSV: {csv_out}  ({len(vol_df)} rows)')

    return arr

print('Cleaning VRAM before embedding extraction...')
for _var in ['optimizer', 'scaler']:
    if _var in dir(): exec(f'del {_var}')
gc.collect(); torch.cuda.empty_cache()
free_gb = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
print(f'VRAM free: {free_gb:.1f} GB')

print('\nExtracting SwinUNETR embeddings v3 (layers3 deeper hook, ROI-crop + octant + volumetric)...')
embeddings = extract_embeddings(model)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# EXTRACTION v4 — Post Option-B (Temporal Contrastive)
# ═══════════════════════════════════════════════════════════════
# Uses the SAME ROI-crop + octant + region + vol strategy (Option A)
# but with the model fine-tuned with temporal contrastive loss.
# Output: vit_swinunetr_embeddings_v4.npz
#         vit_spatial_tokens_v4.npz
#
# If SKIP_OPTION_B=True, this cell is skipped — use extraction v3 output.

if not SKIP_OPTION_B and emb_head is not None:
    # Load best v4 checkpoint
    v4_ckpt = CKPT_DIR / 'swinunetr_best_v4.pth'
    if v4_ckpt.exists():
        ck4 = torch.load(v4_ckpt, map_location=device)
        model.load_state_dict(ck4['model'])
        print(f"  Loaded v4 checkpoint: epoch={ck4.get('epoch','?')} Dice={ck4.get('best_dice',0):.4f}")
    else:
        print("  No v4 checkpoint found — using current model state")

    # Override output filename for v4 inside extract_embeddings
    def extract_v4_wrapper(model):
        """Thin wrapper: calls extract_embeddings but redirects output to v4 filenames."""
        import types
        # Temporarily patch emb_dir filenames inside the function
        emb_dir = OUTPUT_ROOT / 'embeddings'
        original_extract = extract_embeddings

        # Run extraction (layers3 hook already configured)
        print("\nExtracting SwinUNETR embeddings v4 (Option B — temporal contrastive)...")
        embs_arr, ids_arr, tps_arr, sp_tok_arr, n_tok_arr, bbox_arr = [], [], [], [], [], []

        model.eval()
        _feats = {}; hooks_v4 = []
        sv = model.swinViT
        target_layer = 'layers3' if hasattr(sv, 'layers3') else 'layers2'
        C_feat = 384 if target_layer == 'layers3' else 192
        OCT_DIM = 8 * C_feat; REG_DIM = 3 * C_feat; VOL_DIM = 9; TOTAL_DIM = OCT_DIM + REG_DIM + VOL_DIM

        layer_list = getattr(sv, target_layer)
        tgt = layer_list[0] if hasattr(layer_list, '__getitem__') else layer_list
        def _hk(m, inp, out):
            feat = out[-1] if isinstance(out, (list, tuple)) else out
            _feats['feat'] = feat.detach()
        hooks_v4.append(tgt.register_forward_hook(_hk))
        print(f"  Hook: swinViT.{target_layer}[0]  C={C_feat}  dim={TOTAL_DIM}-D")

        all_dicts = train_dicts + val_dicts
        ds = Dataset(all_dicts, val_transforms)
        loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0)
        n_skip = n_empty = 0

        with torch.no_grad():
            for idx, batch in enumerate(safe_emb_iter(loader)):
                pid = batch['patient_id'][0]
                tp  = int(batch['timepoint'][0]) if 'timepoint' in batch else 0
                try:
                    _feats.clear()
                    _ = model(batch['image'].to(device))
                    if 'feat' not in _feats:
                        n_skip += 1; continue

                    feat  = _feats['feat']       # (1, C, H, W, D)
                    C     = feat.shape[1]
                    lbl_p = batch['label'].to(device)  # (1, 3, 128, 128, 128)
                    h, w, d = feat.shape[2:]
                    lbl_feat = torch.nn.functional.adaptive_avg_pool3d(lbl_p, (h, w, d))

                    bbox = get_wt_bbox(lbl_feat[0], min_size=2)
                    if bbox is not None:
                        z0, z1, y0, y1, x0, x1 = bbox
                        feat_crop = feat[:, :, z0:z1, y0:y1, x0:x1]
                        oct_pooled = torch.nn.functional.adaptive_avg_pool3d(feat_crop, (2,2,2))
                        oct_vec = oct_pooled[0].view(C, -1).T.cpu().numpy().flatten()  # (C,8).T=(8,C) → 8C
                        lbl_crop = lbl_feat[:, :, z0:z1, y0:y1, x0:x1]
                        r_vecs = []
                        for ch in range(3):
                            m_ = lbl_crop[0, ch]; s_ = m_.sum()
                            if s_ > 0:
                                r_vecs.append((feat_crop[0] * m_).sum((-3,-2,-1)) / s_)
                            else:
                                r_vecs.append(torch.zeros(C, device=feat.device))
                        reg_vec = torch.cat(r_vecs).cpu().numpy()  # 3C
                        sp_tok  = feat_crop[0].reshape(C, -1).T.cpu().numpy()
                        bbox_entry = list(bbox)
                    else:
                        oct_vec = np.zeros(8*C, dtype=np.float32)
                        reg_vec = np.zeros(3*C, dtype=np.float32)
                        sp_tok  = np.zeros((1, C), dtype=np.float32)
                        bbox_entry = [0,0,0,0,0,0]
                        n_empty += 1

                    # Volumetric features from GT label
                    lbl_np = batch['label'][0].cpu().numpy()  # (3, 128, 128, 128)
                    wt = float(lbl_np[0].sum()); tc = float(lbl_np[1].sum()); et = float(lbl_np[2].sum())
                    vol_vec = np.array([
                        np.log1p(wt), np.log1p(tc), np.log1p(et),
                        float(wt > 0),   float(tc > 0),   float(et > 0),
                        tc / max(wt, 1), et / max(wt, 1), et / max(tc, 1)
                    ], dtype=np.float32)

                    emb = np.concatenate([oct_vec.astype(np.float32),
                                          reg_vec.astype(np.float32),
                                          vol_vec])
                    embs_arr.append(emb); ids_arr.append(pid); tps_arr.append(tp)
                    sp_tok_arr.append(sp_tok); n_tok_arr.append(sp_tok.shape[0])
                    bbox_arr.append(bbox_entry)

                    if idx == 0:
                        print(f"  [FIRST SCAN] {pid} tp={tp}")
                        print(f"    feat shape: {feat.shape}  | oct={len(oct_vec)}  reg={len(reg_vec)}  vol={len(vol_vec)}")
                        print(f"    emb dim: {len(emb)}  | vol_vec: log_wt={vol_vec[0]:.2f} has_wt={vol_vec[3]:.0f}")
                    if (idx+1) % 100 == 0:
                        print(f"  [{idx+1:4d}/{len(all_dicts)}] {pid} tp={tp}")

                except Exception as ex:
                    n_skip += 1
                    if idx < 5: print(f"  [{idx+1}] ERROR {pid}: {ex}")

        for h_ in hooks_v4: h_.remove()

        arr = np.array(embs_arr)
        emb_dir_p = OUTPUT_ROOT / 'embeddings'; emb_dir_p.mkdir(exist_ok=True)
        out = emb_dir_p / 'vit_swinunetr_embeddings_v4.npz'
        np.savez_compressed(out, embeddings=arr,
                            patient_ids=np.array(ids_arr),
                            timepoints=np.array(tps_arr))

        # Save spatial tokens
        max_tok = max(t.shape[0] for t in sp_tok_arr)
        C_tok = sp_tok_arr[0].shape[1]
        padded = np.zeros((len(sp_tok_arr), max_tok, C_tok), dtype=np.float32)
        n_toks = np.zeros(len(sp_tok_arr), dtype=np.int32)
        for j, tok in enumerate(sp_tok_arr):
            padded[j, :tok.shape[0]] = tok; n_toks[j] = tok.shape[0]
        tok_out = emb_dir_p / 'vit_spatial_tokens_v4.npz'
        np.savez_compressed(tok_out, spatial_tokens=padded, token_counts=n_toks,
                            patient_ids=np.array(ids_arr), timepoints=np.array(tps_arr),
                            bboxes=np.array(bbox_arr))

        # Generate tumor_volumes.csv (11-col voxel-based, same schema as v2)
        import pandas as pd
        vol_rows = []
        for j_v, (p_id, t_p, emb_v) in enumerate(zip(ids_arr, tps_arr, embs_arr)):
            v9 = emb_v[-9:]  # last 9 dims = vol features
            wt_v = float(np.expm1(v9[0])); tc_v = float(np.expm1(v9[1])); et_v = float(np.expm1(v9[2]))
            vol_rows.append({
                "patient_id": p_id, "timepoint": int(t_p),
                "wt_vol": wt_v, "tc_vol": tc_v, "et_vol": et_v,
                "has_wt": float(v9[3]>0.5), "has_tc": float(v9[4]>0.5), "has_et": float(v9[5]>0.5),
                "tc_wt_ratio": float(v9[6]), "et_wt_ratio": float(v9[7]), "et_tc_ratio": float(v9[8])
            })
        vol_df = pd.DataFrame(vol_rows)
        vol_csv = emb_dir_p / "tumor_volumes_v4.csv"
        vol_df.to_csv(vol_csv, index=False)
        print(f"  ✅ tumor_volumes_v4.csv: {vol_csv}  ({len(vol_df)} rows, 10 cols)")

        print(f"\n  ✅ v4 embeddings: {out}")
        print(f"     Shape: {arr.shape} | dim={arr.shape[1]}")
        print(f"     Skipped: {n_skip} | Empty ROI: {n_empty}")
        print(f"  ✅ v4 spatial tokens: {tok_out}  shape={padded.shape}")

    extract_v4_wrapper(model)
else:
    print("  Option B skipped — v4 extraction not run")
    print("  Using v3 embeddings (Option A only)")


In [ ]:
# ═══════════ Embedding Diversity Check (v3/v4 aware) ═══════════
# Search: v4 (Option B) → v3 (Option A) → v2 (baseline)
import random as _rnd
import numpy as _np

emb_dir = OUTPUT_ROOT / 'embeddings'
npz_path = None
for fname in ['vit_swinunetr_embeddings_v4.npz',
              'vit_swinunetr_embeddings_v3.npz',
              'vit_swinunetr_embeddings.npz']:
    candidate = emb_dir / fname
    if candidate.exists():
        npz_path = candidate
        print(f'  Found: {fname}')
        break

if npz_path is not None:
    data    = _np.load(npz_path)
    ids_arr = data['patient_ids']
    emb_arr = data['embeddings']
    N, D    = emb_arr.shape
    norms   = _np.linalg.norm(emb_arr, axis=1)
    emb_norm = emb_arr / (norms[:, None] + 1e-8)

    # Sample 500 random pairs (fast even for N=1620)
    n_pairs = min(500, N*(N-1)//2)
    idx_pairs = [(i, j) for i in range(N) for j in range(i+1, N)]
    sampled  = _rnd.sample(idx_pairs, n_pairs)
    sims     = [float(_np.dot(emb_norm[i], emb_norm[j])) for i, j in sampled]
    cos_mean = float(_np.mean(sims))
    diversity = 1.0 - cos_mean

    status = '✅ GOOD' if diversity > 0.20 else ('⚠ MEDIUM' if diversity > 0.10 else '❌ LOW (collapse)')
    print(f'  Embeddings: ({N}, {D})')
    print(f'  Cosine sim (500 pairs): mean={cos_mean:.3f}  std={float(_np.std(sims)):.3f}')
    print(f'  Diversity: {diversity:.3f}  [{status}]')
    print(f'  Unique patients: {len(set(ids_arr))}')

    tps = data['timepoints']
    for tp in sorted(set(tps)):
        idx = [i for i, t in enumerate(tps) if t == tp]
        print(f'  Timepoint {tp}: {len(idx)} scans')

    # Vol features check (last 9 dims)
    vol9 = emb_arr[:, -9:]
    print(f'\n  Vol features (last 9 dims) sanity:')
    print(f'    dim[-9] log_wt: min={vol9[:,0].min():.2f}  max={vol9[:,0].max():.2f}  (expect 0-12)')
    print(f'    dim[-6] has_wt: unique={sorted(set(vol9[:,3].round(1)))}  (expect [0,1])')
else:
    print('  ⚠ No embedding file found yet — run extraction cell first')
    print(f'  Searched in: {emb_dir}')
    try:
        existing = list(emb_dir.glob('*.npz')) if emb_dir.exists() else []
        print(f'  Files present: {[f.name for f in existing]}')
    except Exception:
        pass


In [ ]:
summary = {
    'model': MODEL_NAME, 'best_dice': float(best_dice),
    'regions': REGIONS, 'label': 'BraTS2024: WT=1+2+3(NETC+SNFH+ET), TC=1+3(NETC+ET), ET=3, RC=4(excluded)',
    'train_scans': len(train_dicts), 'val_scans': len(val_dicts),
    'target_brats2021': {'WT': 0.9005, 'TC': 0.8673, 'ET': 0.8509},
    'architecture': 'SwinUNETR(feature_size=48, use_checkpoint=True)',
    'pretrained': 'fold1_f48_ep300_4gpu_dice0_9059',
}
(OUTPUT_ROOT / 'summary.json').write_text(json.dumps(summary, indent=2))

print('='*55)
print(f'  SwinUNETR Fine-Tuning Complete')
print(f'  Best Mean Dice:  {best_dice:.4f}')
print(f'  Regions:         {REGIONS}')
print(f'  Target (BraTS2021): WT=0.900 TC=0.867 ET=0.851')
print(f'  Outputs: {OUTPUT_ROOT}')
print('='*55)
